In [0]:
%sql
CREATE TABLE IF NOT EXISTS company_catalog.company_test.company_docs (
    id BIGINT,
    content STRING,
    category STRING
)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

In [0]:
%sql
INSERT INTO company_catalog.company_test.company_docs VALUES
(1, 'Databricks Vector Search supports semantic retrieval.', 'AI'),
(2, 'Unity Catalog provides centralized governance.', 'Governance'),
(3, 'Delta Lake supports ACID transactions.', 'Data Engineering');

In [0]:
%sql
SHOW TBLPROPERTIES company_catalog.company_test.company_docs;


In [0]:
%pip install databricks-vectorsearch

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

endpoint_name = "company-vs-endpoint"

try:
    vsc.create_endpoint(
        name=endpoint_name,
        endpoint_type="STANDARD"
    )
except Exception as e:
    print(e)

In [0]:
vsc.create_delta_sync_index(
    endpoint_name=endpoint_name,
    index_name="company_catalog.company_test.company_docs_index",
    source_table_name="company_catalog.company_test.company_docs",
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_source_column="content",
    embedding_model_endpoint_name="databricks-gte-large-en"
)

In [0]:
index = vsc.get_index(
    endpoint_name=endpoint_name,
    index_name="company_catalog.company_test.company_docs_index"
)

index.sync()

In [0]:
results = index.similarity_search(
    query_text="How does Databricks support semantic search?",
    columns=["id", "content", "category"],
    num_results=2
)

display(results)

FRESHNESS TEST

In [0]:
%sql
INSERT INTO company_catalog.company_test.company_docs
VALUES
(4,  'Vector indexes are automatically updated using Delta Sync.', 'AI'),
(5,  'Embeddings convert text into numerical vectors for semantic search.', 'AI'),
(6,  'Large language models use retrieved context to generate grounded answers.', 'AI'),
(7,  'Hybrid search combines keyword matching with vector similarity.', 'AI'),
(8,  'Databricks Vector Search supports metadata filtering during retrieval.', 'AI'),
(9,  'Retrieval augmented generation improves answer accuracy using external knowledge.', 'AI'),
(10, 'Similarity search identifies documents with related meaning rather than exact keywords.', 'AI'),
(11, 'Chunking documents into smaller sections can improve retrieval performance.', 'AI'),
(12, 'Vector databases store embeddings to enable semantic search workloads.', 'AI'),
(13, 'Foundation models can generate embeddings from text content.', 'AI'),
(14, 'Unity Catalog provides centralized governance for data and AI assets.', 'Governance'),
(15, 'Row level security can restrict data visibility based on user identity.', 'Governance'),
(16, 'Delta Lake provides ACID transactions and reliable storage.', 'Data Engineering'),
(17, 'Change Data Feed tracks inserts updates and deletes in Delta tables.', 'Data Engineering'),
(18, 'Delta Sync Indexes automatically synchronize changes from Delta tables.', 'AI');

In [0]:
index.sync()

In [0]:
results = index.similarity_search(
    query_text="How are vector indexes updated?",
    columns=["id","content"],
    num_results=1
)

display(results)

FILTERED SEARCH

In [0]:
index.describe()

In [0]:
results = index.similarity_search(
    query_text="semantic search capabilities",
    columns=["id","content","category"],
    num_results=2
)

display(results)

In [0]:
results = index.similarity_search(
    query_text="semantic search capabilities",
    columns=["id","content","category"],
    filters={"category": "AI"},
    num_results=2
)

display(results)

INDEX TYPE COMPARISON

In [0]:
delta_index = vsc.get_index(
    endpoint_name=endpoint_name,
    index_name="company_catalog.company_test.company_docs_index"
)

delta_index.describe()

In [0]:
try:
    vsc.create_direct_access_index(
        endpoint_name=endpoint_name,
        index_name="company_catalog.company_test.docs_direct_access",
        primary_key="id",
        embedding_dimension=1024,
        embedding_vector_column="embedding",
        schema={"id": "integer", "content": "string", "category": "string", "embedding": "array<float>"}
    )
except Exception as e:
    if "already exists" in str(e):
        print(f"Index already exists, skipping creation.")
    else:
        raise

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

response = w.serving_endpoints.query(
    name="databricks-gte-large-en",
    input="Databricks Vector Search supports semantic retrieval."
)

embedding = response.data[0].embedding

In [0]:
records = [
    {
        "id": 1,
        "embedding": embedding,
        "content": "Databricks Vector Search supports semantic retrieval."
    }
]

direct_index = vsc.get_index(
    endpoint_name=endpoint_name,
    index_name="company_catalog.company_test.docs_direct_access"
)

direct_index.upsert(records)

In [0]:
#Delta Sync
delta_index.similarity_search(
    query_text="semantic search",
    columns=["id","content"],
    num_results=2
)

In [0]:
#Direct Access

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
query_text = "How does Databricks support semantic search?"
response = w.serving_endpoints.query(
    name="databricks-gte-large-en",
    input=query_text
)

query_embedding = response.data[0].embedding

direct_index.similarity_search(
    query_vector=query_embedding,
    columns=["id","content"],
    num_results=2
)

In [0]:
direct_index = vsc.get_index(
    endpoint_name="company-vs-endpoint",
    index_name="company_catalog.company_test.docs_direct_access"
)

results = direct_index.similarity_search(
    query_vector=query_embedding,
    columns=["id", "content"],
    num_results=2
)

results

This clearly demonstrates the key difference:

Delta Sync Index → accepts query_text and handles embeddings automatically.
Direct Vector Access Index → requires you to generate embeddings yourself and pass query_vector.

##TRADEOFF SUMMARY

| Area                 | Delta Sync            | Direct Vector Access |
| -------------------- | --------------------- | -------------------- |
| Setup Effort         | Low                   | Higher               |
| Embedding Generation | Automatic             | Manual               |
| Updates              | Automatic via CDF     | Manual upsert        |
| Governance           | Native UC integration | More app-managed     |
| Operational Overhead | Low                   | High                 |
| Flexibility          | Moderate              | High                 |

##Conclusion

The Delta Sync Index was significantly easier to create and maintain because Databricks automatically synchronized data changes and generated embeddings from the source Delta table. The Direct Vector Access Index required additional steps for embedding generation and manual vector upserts but provided greater control over how vectors were created, updated, and managed. For continuously changing Delta data, Delta Sync was the simpler solution, while Direct Vector Access offered maximum flexibility for custom or external vector pipelines.

##Query Tuning

In [0]:
results_top3 = index.similarity_search(
    query_text="How does Databricks support semantic search?",
    columns=["id", "content", "category"],
    num_results=3
)

display(results_top3)

In [0]:
results_top15 = index.similarity_search(
    query_text="How does Databricks support semantic search?",
    columns=["id", "content", "category"],
    num_results=15
)

display(results_top15)

#Results
top_k=3

Returned the three most relevant chunks related to:

Vector Search
Embeddings
Delta Sync

The results were highly focused and directly answered the query.

top_k=15

Returned the same highly relevant chunks plus additional documents about:

Unity Catalog
Delta Lake
Governance
Other platform concepts

While broader, some results were less relevant to the query.

###Observation

Increasing top_k improved recall by returning more potentially relevant documents, but also introduced additional noise. The larger result set contained documents that were related to Databricks but not directly useful for answering the specific question.

###Conclusion

For a production RAG application, I would start with top_k=5 to 10. In this experiment, top_k=3 produced the most focused results, while top_k=15 increased coverage at the cost of relevance and token usage. The appropriate value depends on the complexity of the question and the size of the document corpus, but a moderate top_k typically provides the best balance between recall and precision.